# Cross-branch top-up — A→B judge anchors + direction decomposition

Two small jobs, **~45 min total**, both independently banked. Runs comfortably
while you annotate.

| § | job | units | ~T4 |
|---|---|---|---|
| A | judge A→B B2/B3 (baseline + reference) on quadrant C | — | ~18 min |
| B | Δ_A^∥ / Δ_A^⊥ decomposition: build + generate 2 arms (A→B, coef 1.0) | 2 | ~20 min |
| C | judge the 2 decomposition arms on quadrant C *(optional)* | — | ~10 min |

**Why A:** the earlier A→B judge run scored only the 6 Stage-2 arms; without the
B2/B3 anchors the A→B judge analysis can only do arm-vs-arm contrasts. This adds
the two model conditions so A→B matches B→A.

**Why B:** "the transferred component is richer than the generic refusal
direction" currently rests on one contrast (identity − dir_source). The
decomposition splits Δ_A along the source refusal direction into a parallel
part and a residual, **each rescaled per row back to ‖Δ_A(x)‖** so both inject
the same magnitude as identity and differ only in direction. If the parallel
arm alone reproduces the transfer and the residual does not → the transferable
signal *is* essentially the refusal direction (just needs full magnitude). If
the residual also moves C → there is transferable structure beyond it.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO_URL='https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR='/content/dpo-safety-representations'
BRANCH='agent/c-quadrant-end-to-end-e0e2317a'
PINNED='c784139b156910b717a73ae7ae5e1f2cca803c98'
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","-b",BRANCH,REPO_URL,REPO_DIR],check=True)
os.chdir(REPO_DIR)
subprocess.run(["git","fetch","origin"],check=True)
subprocess.run(["git","checkout",PINNED],check=True)
print("HEAD",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())

### 0.2 HF auth — REQUIRED (both judge sections)

In [ ]:
import os
try:
    from google.colab import userdata
    from huggingface_hub import login
    _t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=_t; login(token=_t)
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:',repr(e))
    print('Section B (decomposition generation) still runs; A and C need the token.')

### 0.3 Apply the patch — upload the LATEST `crossbranch_p0_patch.zip`
(the build with `decompose_dosematched`, the parallel/perp conditions, and `build_judge_manifest --conditions`)

In [ ]:
from google.colab import files
import zipfile, io
up=files.upload(); assert len(up)==1,"upload crossbranch_p0_patch.zip"
name,data=next(iter(up.items()))
with zipfile.ZipFile(io.BytesIO(data)) as z:
    ns=z.namelist()
    assert all(n.startswith(('src/crossbranch/','tests/crossbranch/')) for n in ns)
    z.extractall('.')
print("applied",len(ns),"files")

In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes
!pip uninstall -y torchao || true
!nvidia-smi

### 0.5 Restore prior results — upload every `crossbranch_finish_*.zip` you have (and `crossbranch_prior_results.zip`). Multi-select is fine.

In [ ]:
import zipfile, io, os, glob
from google.colab import files
os.makedirs("results/crossbranch", exist_ok=True)
print("upload every result zip you have (crossbranch_finish_*, crossbranch_prior_results, crossbranch_topup_*); cancel to skip:")
for name, data in files.upload().items():
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall("results/crossbranch")
    print("restored", name)
need = ["crossbranch_AtoB_baseline_target_coefna.json",
        "crossbranch_AtoB_reference_target_coefna.json",
        "crossbranch_AtoB_xfer_delta_source_identity_coef1.json"]
for n in need:
    print(("  OK  " if os.path.exists("results/crossbranch/raw/" + n) else "  MISSING  ") + n)

### 0.6 Copy activations + directions from Drive

In [ ]:
RESULTS_SOURCE_DIR='/content/drive/MyDrive/dpo_v2/results'
import shutil
from pathlib import Path
src=Path(RESULTS_SOURCE_DIR); assert (src/'activations').exists(), f"fix RESULTS_SOURCE_DIR"
copied,missing=[],[]
d=Path('results/activations'); d.mkdir(parents=True,exist_ok=True)
for s in ('M2','M3','M2_alt','M3_alt'):
    for suf in ('_final.npy','_pooled.npy','_metadata.json','_metadata_binding.json'):
        f=src/'activations'/f'{s}{suf}'; (copied if f.exists() else missing).append(f.name)
        if f.exists(): shutil.copy2(f,d/f'{s}{suf}')
dd=Path('results/refusal_direction'); dd.mkdir(parents=True,exist_ok=True)
for s in ('M3','M3_alt'):
    for suf in ('_direction_654.npy','_direction_654_binding.json'):
        f=src/'refusal_direction'/f'{s}{suf}'; (copied if f.exists() else missing).append(f.name)
        if f.exists(): shutil.copy2(f,dd/f'{s}{suf}')
print("copied",len(copied)); [print("  MISSING",m) for m in missing]
assert not missing

In [ ]:
!python -m pytest tests/crossbranch -q

---
# SECTION A — A→B judge anchors (B2 / B3) on quadrant C  (~18 min)

Builds a manifest restricted to `baseline_target` + `reference_target` for A→B,
quadrant C (208 rows), and scores them with both judges. `--resume-from` merges
into the existing A→B judge file so the 6 Stage-2 arms already scored are kept.

In [ ]:
import glob, json, pathlib, subprocess, sys

MAN = "results/crossbranch/manifests/crossbranch_judge_manifest.json"
JDIR = "results/crossbranch/judges"

subprocess.run([sys.executable, "-m", "src.crossbranch.build_judge_manifest",
                "--quadrants", "C", "--directions", "AtoB",
                "--conditions", "baseline_target", "reference_target"], check=True)

def newest_after(before):
    new = sorted(set(glob.glob(f"{JDIR}/behavioral_judges_*.json")) - before)
    if not new:
        raise RuntimeError("behavioral_judges did not write a new output file")
    return new[-1]

# The one pre-existing A->B judge file (6 Stage-2 arms, no baseline/reference).
# NOTE: passing it as --resume-from does NOT merge it into the new output --
# behavioral_judges only carries forward SCORES for records that already
# appear in the CURRENT manifest, it never appends records the manifest
# doesn't have. Since the 6 old arms aren't in this 208-row manifest at all,
# a plain --resume-from run would silently produce a "_full" file containing
# only the 2 new arms. So the merge below is done explicitly, by hand.
prev = sorted(glob.glob(f"{JDIR}/behavioral_judges_*AtoB*.json"))
old_file = prev[-1] if prev else None

# Pass 1: StrongREJECT only. Writes to disk BEFORE WildGuard ever runs.
# behavioral_judges.py's score_wildguard() has no try/except around
# raw_generate() (unlike score_1_to_5, which does) and the whole file is only
# written once, at the very end of run_judges -- so a single bad WildGuard
# generation can crash the process and lose every score computed so far in
# the same invocation. Frozen file, can't patch that there; splitting into
# two invocations means a WildGuard failure can only ever cost pass 2, never
# pass 1's already-written strong_reject scores.
before = set(glob.glob(f"{JDIR}/behavioral_judges_*.json"))
cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
       "--response-manifest", MAN, "--out-dir", JDIR,
       "--run-live", "--scope", "all", "--skip-wildguard"]
subprocess.run(cmd, check=True)
sr_out = newest_after(before)
print("pass 1 (strong_reject) wrote:", sr_out)

# Pass 2: WildGuard, resuming from pass 1's own output so strong_reject scores
# carry forward and only wildguard is computed fresh.
before = set(glob.glob(f"{JDIR}/behavioral_judges_*.json"))
cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
       "--response-manifest", MAN, "--out-dir", JDIR,
       "--run-live", "--scope", "all", "--resume-from", sr_out]
subprocess.run(cmd, check=True)
new_out = newest_after(before)
print("pass 2 (wildguard) wrote:", new_out)

# Explicit merge: old 6-arm file's records + new 2-arm file's records, so the
# final _AtoB_full.json genuinely carries all 8 arms in one file (what the
# cell's job has always been meant to produce).
new_data = json.loads(pathlib.Path(new_out).read_text(encoding="utf-8"))
if old_file:
    old_data = json.loads(pathlib.Path(old_file).read_text(encoding="utf-8"))
    assert old_data.get("benchmark_sha256") == new_data.get("benchmark_sha256"), \
        "old/new judge files bind to different benchmark SHAs -- not safe to merge"
    merged = dict(new_data)
    merged["records"] = old_data.get("records", []) + new_data.get("records", [])
    merged["n_records"] = len(merged["records"])
    merged["merged_from"] = [old_file, new_out]
    print(f"merged {len(old_data.get('records', []))} old + "
          f"{len(new_data.get('records', []))} new = {merged['n_records']} records")
else:
    merged = new_data
    print("no pre-existing A->B judge file found -- nothing to merge")

tagged = new_out.replace(".json", "_AtoB_full.json")
pathlib.Path(tagged).write_text(json.dumps(merged, indent=2), encoding="utf-8")
print("wrote merged:", tagged)

### BANK A

In [ ]:
import shutil, os
from google.colab import files
stage = "/content/_bank"
if os.path.isdir(stage):
    shutil.rmtree(stage)
for sub in ("raw", "analysis", "judges", "manifests"):
    d = "results/crossbranch/" + sub
    if os.path.isdir(d):
        shutil.copytree(d, stage + "/" + sub, ignore=shutil.ignore_patterns("shards"))
z = shutil.make_archive("/content/crossbranch_topup_A", "zip", stage)
print("crossbranch_topup_A.zip", round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    files.download(z)
except Exception as e:
    print("(download failed:", e, "-- re-run this cell)")

---
# SECTION B — Δ_A^∥ / Δ_A^⊥ decomposition  (~20 min, 2 units)

CPU build reuses `delta_source` (already in the restore); generation is 2
arms × 414 rows into M2_alt, coef 1.0.

In [ ]:
!python -m src.crossbranch.delta --decomposition --source-branch A --target-branch B --out-dir results/crossbranch/deltas

In [ ]:
!python -m src.crossbranch.runner --dry-run --allow-stage2 \
    --conditions xfer_delta_source_parallel xfer_delta_source_perp --coefficients 1.0

In [ ]:
import subprocess, sys
for cond in ("xfer_delta_source_parallel", "xfer_delta_source_perp"):
    print("generating", cond, flush=True)
    subprocess.run([sys.executable, "-m", "src.crossbranch.runner",
                    "--allow-stage2", "--conditions", cond,
                    "--coefficients", "1.0"], check=True)
print("decomposition arms done -- bank in the next cell")

### BANK B

In [ ]:
import shutil, os
from google.colab import files
stage = "/content/_bank"
if os.path.isdir(stage):
    shutil.rmtree(stage)
for sub in ("raw", "analysis", "judges", "manifests"):
    d = "results/crossbranch/" + sub
    if os.path.isdir(d):
        shutil.copytree(d, stage + "/" + sub, ignore=shutil.ignore_patterns("shards"))
z = shutil.make_archive("/content/crossbranch_topup_B", "zip", stage)
print("crossbranch_topup_B.zip", round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    files.download(z)
except Exception as e:
    print("(download failed:", e, "-- re-run this cell)")

---
# SECTION C — judge the decomposition arms on quadrant C  (~10 min, OPTIONAL)

Only run if you want the parallel/perp comparison on the continuous judges too.
Skip it and the rule-based `analyze_stage2` still gives the decomposition
result.

In [ ]:
import glob, json, pathlib, subprocess, sys

MAN = "results/crossbranch/manifests/crossbranch_judge_manifest.json"
JDIR = "results/crossbranch/judges"

subprocess.run([sys.executable, "-m", "src.crossbranch.build_judge_manifest",
                "--quadrants", "C", "--directions", "AtoB",
                "--conditions", "xfer_delta_source_parallel", "xfer_delta_source_perp"],
               check=True)

def newest_after(before):
    new = sorted(set(glob.glob(f"{JDIR}/behavioral_judges_*.json")) - before)
    if not new:
        raise RuntimeError("behavioral_judges did not write a new output file")
    return new[-1]

# Same two-pass split as SECTION A: score_wildguard() has no try/except around
# raw_generate() and the file is only written once at the end of run_judges,
# so a bad WildGuard generation would otherwise lose the strong_reject scores
# from this same invocation too.
before = set(glob.glob(f"{JDIR}/behavioral_judges_*.json"))
subprocess.run([sys.executable, "-m", "src.analysis.behavioral_judges",
                "--response-manifest", MAN, "--out-dir", JDIR,
                "--run-live", "--scope", "all", "--skip-wildguard"], check=True)
sr_out = newest_after(before)
print("pass 1 (strong_reject) wrote:", sr_out)

before = set(glob.glob(f"{JDIR}/behavioral_judges_*.json"))
subprocess.run([sys.executable, "-m", "src.analysis.behavioral_judges",
                "--response-manifest", MAN, "--out-dir", JDIR,
                "--run-live", "--scope", "all", "--resume-from", sr_out], check=True)
new_out = newest_after(before)

tagged = new_out.replace(".json", "_AtoB_decomp.json")
pathlib.Path(new_out).rename(tagged)
print("tagged:", tagged)

### BANK C (final)

In [ ]:
import shutil, os
from google.colab import files
stage = "/content/_bank"
if os.path.isdir(stage):
    shutil.rmtree(stage)
for sub in ("raw", "analysis", "judges", "manifests"):
    d = "results/crossbranch/" + sub
    if os.path.isdir(d):
        shutil.copytree(d, stage + "/" + sub, ignore=shutil.ignore_patterns("shards"))
z = shutil.make_archive("/content/crossbranch_topup_all", "zip", stage)
print("crossbranch_topup_all.zip", round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    files.download(z)
except Exception as e:
    print("(download failed:", e, "-- re-run this cell)")

---
## STOP — local CPU

Bring back the latest `crossbranch_topup_*.zip`, unzip into
`results/crossbranch/`, then:

```
# decomposition analysis (rule-based) — parallel/perp fold into analyze_stage2 automatically
python -m src.crossbranch.analyze_stage2 --coef 1.0
python -m src.crossbranch.plot_stage2

# A->B judge analysis, now with anchors
python -m src.crossbranch.analyze_judges \
    --judge-file results/crossbranch/judges/behavioral_judges_<ts>_AtoB_full.json

# (if section C ran) decomposition judges
python -m src.crossbranch.analyze_judges \
    --judge-file results/crossbranch/judges/behavioral_judges_<ts>_AtoB_decomp.json \
    --out-name crossbranch_judge_analysis_AtoB_decomp.json
```

Expected: `crossbranch_AtoB_stage2_analysis.json` now carries
`xfer_delta_source_parallel` / `xfer_delta_source_perp` arms and the
`identity − parallel`, `identity − perp`, `parallel − perp` contrasts; the
A→B judge analysis has non-empty `by_direction_and_coef` with a baseline anchor.